# Single Asset Analyzer
Γράψε ticker στο κελί 2 και τρέξε από πάνω προς τα κάτω.
Δοκιμές, επισκευές και έλεγχοι ζουν στον φάκελο `dev/`, όχι εδώ.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
# DEV_MODE = True μόνο όταν μόλις άλλαξες κώδικα στο quality_scores/
# (ξαναφορτώνει τα modules χωρίς restart). Για κανονική χρήση: False.
DEV_MODE = True

import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

if DEV_MODE:
    import importlib
    import db as _db; importlib.reload(_db)
    import fundamentals as _f; importlib.reload(_f)
    from quality_scores import line_items as _li; importlib.reload(_li)
    from quality_scores import piotroski as _pi; importlib.reload(_pi)
    from quality_scores import piotroski_plots as _pp; importlib.reload(_pp)
    from valuation import pe as _pe; importlib.reload(_pe)
    from valuation import pe_plots as _pep; importlib.reload(_pep)
    from quality_scores import diagnostics as _D; importlib.reload(_D)

from db import get_engine
from quality_scores.piotroski import piotroski_f_score, print_report, piotroski_history
from quality_scores.piotroski_plots import plot_report, plot_history
from valuation.pe import pe_history, print_pe_summary, print_peer_coverage, print_band_summary, print_decomposition
from valuation.pe_plots import plot_pe, plot_pe_bands, plot_decomposition
from valuation.pe import debug_ttm
from quality_scores import diagnostics as D

eng = get_engine()
print(f"✅ Έτοιμο | DEV_MODE={'ON' if DEV_MODE else 'OFF'}")


In [ ]:
D.integrity_report(eng)

In [ ]:
with eng.connect() as c:
    by_day = pd.read_sql(text("""
        SELECT inserted_at::date AS day, COUNT(*) AS rows, COUNT(DISTINCT ticker) AS tickers
        FROM fundamentals WHERE inserted_at >= '2026-10-10'
        GROUP BY 1 ORDER BY 1"""), c)

    # Από τις γραμμές που μπήκαν από τις 12/10 και μετά: πόσες είναι «στήλες σύγκρισης» (lag > 300);
    bad_new = pd.read_sql(text("""
        SELECT ft.name AS form, COUNT(*) AS rows, COUNT(DISTINCT f.ticker) AS tickers,
               SUM(CASE WHEN f.filed_date - f.period_end > 300 THEN 1 ELSE 0 END) AS lag_gt_300
        FROM fundamentals f JOIN form_type_lookup ft ON ft.id = f.form_type_id
        WHERE f.inserted_at >= '2026-10-12'
        GROUP BY 1 ORDER BY 2 DESC"""), c)

print(by_day.to_string(index=False))
print()
print(bad_new.to_string(index=False))

In [ ]:
TICKER = "MSFT"
AS_OF = None  # π.χ. "2022-06-30" για "ό,τι ήξερε η αγορά τότε"


In [ ]:
from valuation.pe import debug_ttm
for t in ["APH", "ACN", "NFLX", "JNJ", "CDNS"]:
    debug_ttm(t, eng, since="2024-06-01")
    print("\n" + "═" * 70 + "\n")

In [ ]:
import pandas as pd
from sqlalchemy import text

sql = text("""
    SELECT f.ticker, f.fiscal_year, f.fiscal_period,
           f.period_start, f.period_end, f.filed_date, f.value
    FROM fundamentals f
    JOIN metric_lookup m     ON m.id  = f.metric_id
    JOIN form_type_lookup ft ON ft.id = f.form_type_id
    WHERE m.name = 'EarningsPerShareDiluted' AND ft.name = '10-Q'
      AND f.ticker = ANY(:t) AND f.fiscal_year BETWEEN 2024 AND 2026
    ORDER BY f.ticker, f.fiscal_year, f.fiscal_period
""")
with eng.connect() as c:
    print(pd.read_sql(sql, c, params={"t": ["ACN", "APH", "CDNS"]}).to_string(index=False))

# Πόσο μεγάλο είναι το πρόβλημα σε όλη τη βάση (όλα τα metrics, όχι μόνο EPS):
sql2 = text("""
    SELECT COUNT(*) AS groups, COUNT(DISTINCT ticker) AS tickers
    FROM (
        SELECT ticker, metric_id, form_type_id, period_end
        FROM fundamentals
        WHERE period_start IS NOT NULL
        GROUP BY ticker, metric_id, form_type_id, period_end
        HAVING COUNT(DISTINCT fiscal_year) > 1
    ) x
""")
with eng.connect() as c:
    print(c.execute(sql2).fetchone())

# PIOTROSKI F-SCORE
## ΤΙ ΕΙΝΑΙ:
9 ερωτήσεις "ναι/όχι" για την οικονομική υγεία μιας εταιρείας,
συγκρίνοντας την τελευταία χρονιά (φέτος) με την προηγούμενη (πέρσι).
Κάθε "ναι" = 1 πόντος. Γενικά: 7–9 = υγιής, 0–3 = αδύναμη.
Αν λείπει κάποιο νούμερο, η ερώτηση ΔΕΝ μετράει ως αποτυχία — γράφεται
"δεν αξιολογήθηκε" και το score δίνεται ως "πόντοι / όσα ελέγχθηκαν".

## - ΚΕΡΔΟΦΟΡΙΑ -

## 1. Κερδοφόρα (ROA > 0)
    Έβγαλε κέρδος φέτος;  ROA = κέρδη / assets στην ΑΡΧΗ της χρονιάς.
    Παράδειγμα: κέρδη 10, assets αρχής 100 → ROA 10% → ΝΑΙ.

## 2. Θετικές ταμειακές ροές (CFO > 0)
    Μπήκαν πραγματικά χρήματα στο ταμείο από τη λειτουργία της εταιρείας;
    τα κέρδη μπορεί να είναι "στα χαρτιά" — το ταμείο δεν ξεγελιέται εύκολα.

## 3. ROA βελτιώθηκε
    Η κερδοφορία ανά ευρώ assets ανέβηκε σε σχέση με πέρσι;
    Παράδειγμα: πέρσι 9%, φέτος 11% → ΝΑΙ.
    Γιατί assets ΑΡΧΗΣ χρονιάς: αν η εταιρεία αγόρασε άλλη τον Δεκέμβριο
    και τα assets διπλασιάστηκαν, αυτά δεν δούλεψαν όλη τη χρονιά. Με τα
    assets του τέλους θα φαινόταν ψευδώς ότι η κερδοφορία έπεσε στο μισό.

## 4. Ταμείο > λογιστικά κέρδη
    Οι ταμειακές ροές είναι μεγαλύτερες από τα κέρδη;
    Παράδειγμα: κέρδη 10, ταμείο 14 → ΝΑΙ (τα κέρδη "στηρίζονται" σε
    μετρητά). Αν ήταν κέρδη 10, ταμείο 3 → ΟΧΙ: μεγάλο μέρος των κερδών
    είναι απαιτήσεις/λογιστικές εγγραφές, κλασικό προειδοποιητικό σημάδι.

## - ΧΡΕΟΣ & ΡΕΥΣΤΟΤΗΤΑ - 

## 5. Λιγότερο χρέος
    Μακροπρόθεσμο χρέος / ΜΕΣΟ ΟΡΟ assets έπεσε σε σχέση με πέρσι;
    Παράδειγμα: πέρσι 25%, φέτος 20% → ΝΑΙ.
    Συγκρίνεται πάντα το ΙΔΙΟ μέτρο και τις δύο χρονιές: αν πέρσι
    δηλώθηκαν δύο κομμάτια χρέους (30 + 70) και φέτος μόνο το ένα (70),
    συγκρίνουμε 70 με 70 — όχι 100 με 70, που θα έδειχνε ψεύτικη μείωση.

## 6. Καλύτερη ρευστότητα (current ratio)
    Current ratio = ό,τι γίνεται μετρητά μέσα σε 1 χρόνο / ό,τι πρέπει
    να πληρωθεί μέσα σε 1 χρόνο. Ανέβηκε σε σχέση με πέρσι;
    Παράδειγμα: πέρσι 1.40x, φέτος 1.25x → ΟΧΙ.

## 7. Όχι νέες μετοχές
    Ο αριθμός μετοχών έμεινε ίδιος ή μειώθηκε; Νέες μετοχές = η πίτα
    μοιράζεται σε περισσότερους (dilution). Buybacks = ΝΑΙ.
    Αν ο αριθμός άλλαξε κατά ~x2, x4, x10 κτλ. είναι πιθανότατα split
    (π.χ. AAPL 4:1 το 2020: 4.4B → 17B μετοχές) → "δεν αξιολογήθηκε",
    γιατί το split δεν είναι dilution.

## - ΑΠΟΔΟΤΙΚΟΤΗΤΑ - 

## 8. Μεικτό περιθώριο ↑
    Από κάθε 100€ πωλήσεων, πόσα μένουν μετά το κόστος του προϊόντος;
    Παράδειγμα: πέρσι 37.8€, φέτος 38.2€ → ΝΑΙ (καλύτερη τιμολόγηση ή
    φθηνότερη παραγωγή).

## 9. Αποδοτικότητα assets ↑ (asset turnover)
    Πόσες πωλήσεις βγάζει κάθε 1€ assets (αρχής χρονιάς);
    Παράδειγμα: πέρσι 0.71€, φέτος 0.81€ → ΝΑΙ (τα ίδια assets "δουλεύουν" περισσότερο).


In [ ]:
# ── PIOTROSKI F-SCORE ────────────────────────────────────────────────────
res = piotroski_f_score(TICKER, eng, as_of=AS_OF)
print_report(res)
plot_report(res)
plt.show()

plot_history(piotroski_history(TICKER, eng, n_years=10), TICKER)
plt.show()


# P/E — ΦΘΗΝΗ Η ΑΚΡΙΒΗ;
## ΤΙ ΕΙΝΑΙ:
P/E = τιμή μετοχής / κέρδη ανά μετοχή των τελευταίων 12 μηνών (EPS TTM).
Δείχνει πόσα πληρώνεις για κάθε 1 ετήσιου κέρδους.
Παράδειγμα: τιμή 100, EPS TTM 5 → P/E 20 → "πληρώνεις 20 χρόνια κερδών".

## Πώς διαβάζεται το διάγραμμα
    Πάνω: η τιμή. Κάτω: το P/E της μετοχής (μπλε) μαζί με:
    - τη διακεκομμένη μπλε = ο "συνήθης" P/E της ίδιας στην περίοδο
      (μπλε γραμμή πάνω της → πιο ακριβή από ό,τι συνήθως),
    - την πορτοκαλί = διάμεσος P/E του κλάδου,
    - την πορτοκαλί ζώνη = εκεί βρίσκεται η "μεσαία" μισή του κλάδου.
    Γκρι φόντο = η εταιρεία είχε ζημιές → το P/E δεν ορίζεται.

## Προσοχή στην ερμηνεία
    Χαμηλό P/E ≠ αυτόματα ευκαιρία: μπορεί η αγορά να περιμένει πτώση κερδών.
    Υψηλό P/E ≠ αυτόματα φούσκα: μπορεί να περιμένει γρήγορη ανάπτυξη.
    Το P/E λέει "πόσο ακριβά πληρώνεις", όχι "αν αξίζει".


In [ ]:
# ── P/E: ΤΙΜΗ & ΑΠΟΤΙΜΗΣΗ ─────────────────────────────────────────────────
PEER_GROUP = "sector"   # ή "industry" (στενότερη ομάδα, λιγότερες εταιρείες)
PE_YEARS = 10

pe = pe_history(TICKER, eng, years=PE_YEARS, as_of=AS_OF, peer_group=PEER_GROUP)
print_peer_coverage(pe)
print_pe_summary(pe)
plot_pe(pe)
plt.show()


# P/E BANDS — ΠΟΥ ΘΑ ΗΤΑΝ Η ΤΙΜΗ ΣΤΟ "ΣΥΝΗΘΙΣΜΕΝΟ" P/E;
## ΤΙ ΕΙΝΑΙ:
Τιμή στο P/E X = EPS TTM × X. Αν EPS = 5 και ο συνήθης P/E της μετοχής είναι 20, η "λογική" τιμή είναι 100.
Η πράσινη ζώνη = P/E ανάμεσα στο 25% και 75% των ημερών της ίδιας της μετοχής.

    Μπλε (τιμή) κάτω από τη ζώνη → φθηνή σε σχέση με το δικό της παρελθόν
    Μπλε μέσα στη ζώνη           → κανονική αποτίμηση
    Μπλε πάνω από τη ζώνη        → ακριβή σε σχέση με το δικό της παρελθόν

Προσοχή: τα όρια υπολογίζονται από ΟΛΟ το διάστημα (και το μέλλον κάθε ημέρας) — περιγραφικό διάγραμμα, όχι σήμα για backtest.


In [ ]:
# ── P/E BANDS ──────────────────────────────────────────────────────────────
print_band_summary(pe)
plot_pe_bands(pe)
plt.show()


# ΑΠΟ ΠΟΥ ΗΡΘΕ Η ΑΠΟΔΟΣΗ — ΚΕΡΔΗ Η P/E;
## ΤΙ ΕΙΝΑΙ:
Τιμή = EPS × P/E, άρα μεταβολή τιμής = μεταβολή κερδών × μεταβολή P/E.
Παράδειγμα: τιμή 50$→100$ (×2) = κέρδη 2.5→4 (×1.6) × P/E 20→25 (×1.25).

    Άνοδος από κέρδη → η εταιρεία έγινε πραγματικά καλύτερη (πιο "υγιής" άνοδος)
    Άνοδος από P/E   → η αγορά απλώς πληρώνει ακριβότερα· μπορεί να γυρίσει πίσω

Χωρίς μερίσματα. Έτη με EPS κοντά στο 0 εμφανίζονται με "—" (ο διαχωρισμός δεν έχει νόημα).


In [ ]:
# ── ΑΝΑΛΥΣΗ ΑΠΟΔΟΣΗΣ ──────────────────────────────────────────────────────
print_decomposition(pe)
plot_decomposition(pe)
plt.show()


## Βασικές μετρικές
*(Βήμα 2 — έρχεται με το `metrics.py`)*

In [ ]:
TICKER = "MSFT"
pe = pe_history(TICKER, eng, years=10, peer_group="sector")
print_peer_coverage(pe)
print_pe_summary(pe)

print_report(piotroski_f_score(TICKER, eng))